In [78]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from collections import Counter

In [79]:
df = pd.read_csv('../0.dataset/dataset_penjualan_emas_kotor.csv')
df.head(5)

,Transaction_ID,Date,Customer_ID,Gold_Type,Karat,Weight_Gram,Price_Per_Gram,Total_Price,Payment_Method,Store_Location
0,TXN-0295,2024-04-21,CUST-480,Perhiasan,24,35.55,1370210,48710965,Kartu Kredit,JKT Pusat
1,TXN-0809,2024-03-09,CUST-373,Perhiasan,18,NaN,961081,11215815,Transfer Bank,Makassar
2,TXN-0364,2024-11-18,CUST-175,Perhiasan,24,49.59,1323332,65624033,QRIS,Bandung
3,TXN-0869,2025-09-30,CUST-313,Perhiasan,24,44.73,1352769,60509357,QRIS,Makassar
4,TXN-0326,2025-03-13,CUST-158,Koin,18,10.93,960581,10499150,Transfer Bank,Surabaya


In [80]:
df = df.dropna()
df.isnull().sum()

Transaction_ID    0
Date              0
Customer_ID       0
Gold_Type         0
Karat             0
Weight_Gram       0
Price_Per_Gram    0
Total_Price       0
Payment_Method    0
Store_Location    0
dtype: int64

In [81]:
df = df.drop_duplicates(keep='last')
df.duplicated().sum()

np.int64(0)

In [82]:
df = df.drop(columns=['Transaction_ID','Customer_ID','Date'])
df.head()

,Gold_Type,Karat,Weight_Gram,Price_Per_Gram,Total_Price,Payment_Method,Store_Location
0,Perhiasan,24,35.55,1370210,48710965,Kartu Kredit,JKT Pusat
2,Perhiasan,24,49.59,1323332,65624033,QRIS,Bandung
3,Perhiasan,24,44.73,1352769,60509357,QRIS,Makassar
5,Koin,24,38.79,1350607,52390045,Kartu Kredit,Makassar
6,Batangan,24,34.33,1362814,46785404,QRIS,Medan


In [83]:
df_x = df.drop(columns='Gold_Type')
df_y = df[['Gold_Type']]
X_train,X_test,y_train,y_test = train_test_split(df_x,df_y,test_size=0.2,random_state=42)

## 1.Teknik Resampling Data

In [84]:
def compare_smote_rows(x_resampled, y_resampled,x_train = X_train, y_train = y_train):
    comparison_df = pd.DataFrame({
        'Sebelum_Imbalance': [
            x_train.shape[0],
            y_train.shape[0]
        ],
        'Sesudah_Imbalance': [
            x_resampled.shape[0],
            y_resampled.shape[0]
        ]
    }, index=['X', 'y'])

    return comparison_df

def compare_df_y(y_train_resample,y_train = y_train):
    comparison_df = pd.DataFrame({
        'Sebelum_Imbalance_y': y_train.value_counts().sort_index(),
        'Sesudah_Imbalance_y': y_train_resample.value_counts().sort_index()
    })

    return comparison_df

### 1.Random Oversamping

In [85]:
from imblearn.over_sampling import RandomOverSampler

ros = RandomOverSampler(random_state=42)
X_train_resample,y_train_resample = ros.fit_resample(X_train,y_train)

compare_df_y(y_train_resample)

,Sebelum_Imbalance_y,Sesudah_Imbalance_y
Gold_Type,,
Batangan,237,359
Koin,134,359
Perhiasan,359,359


### 2.Random Undersampling

In [86]:
from imblearn.under_sampling import RandomUnderSampler

rus = RandomUnderSampler(random_state=42)
X_train_resample,y_train_resample = rus.fit_resample(X_train,y_train)

compare_df_y(y_train_resample)

,Sebelum_Imbalance_y,Sesudah_Imbalance_y
Gold_Type,,
Batangan,237,134
Koin,134,134
Perhiasan,359,134


### 3.SMOTE Imbalance

In [93]:
from imblearn.over_sampling import SMOTE
from sklearn.preprocessing import LabelEncoder

#data cetegori X harus di encoding dulu
X_train_encoded = X_train.copy()
for col in X_train_encoded.select_dtypes(include='object').columns:
    le = LabelEncoder()
    X_train_encoded[col] = le.fit_transform(X_train_encoded[col])

smote = SMOTE(random_state=42)
X_train_resample,y_train_resample = smote.fit_resample(X_train_encoded,y_train)

compare_df_y(y_train_resample)

,Sebelum_Imbalance_y,Sesudah_Imbalance_y
Gold_Type,,
Batangan,237,359
Koin,134,359
Perhiasan,359,359


## 2.Algoritma Ensemble Learning